In [ ]:
##%%
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Force legacy Keras for TensorFlow Probability compatibility
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras import layers, models, callbacks
import datetime
import time
from pathlib import Path

tfd = tfp.distributions

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Available: {gpus[0]}")
else:
    print("⚠️ GPU not available")
print("="*60)

# Polars configuration
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)



In [ ]:
# ============================================
# NOTEBOOK: WALK-FORWARD TRIAD(1 SEED + HYBRID PARAMS + TIME FEATURES)
# ============================================

import sys

# ============================================
# TIMER - START
# ============================================
import time
notebook_start_time = time.time()
print(f"⏱️ Notebook started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================
# SECTION 1: IMPORTS + CONFIG
# ============================================
import numpy as np
import polars as pl
from pathlib import Path
import os
import time
import datetime
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# ============================================
# VERSION CHECK (required by competition rules)
# ============================================
print("="*60)
print("ENVIRONMENT VERSIONS")
print("="*60)
print(f"Python version: {sys.version}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Polars version: {pl.__version__}")
print(f"NumPy version: {np.__version__}")
print("="*60)

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)

# ============================================
# KONFIGURACJA
# ============================================

# Okna walk-forward
WINDOW_SIZE = 500
STEP = 100

# Scope
MIN_TS = 1
MAX_TS_TRAIN = 3601

# Only 1 seed (instead 3)
SEEDS = [42]

# Horizonts
HORIZONS = [1, 3, 10, 25]

print("="*60)
print("CONFIGURATION OF WALK-FORWARD (1 SEED + HYBRID PARAMS)")
print("="*60)
print(f"Window size: {WINDOW_SIZE}, Step: {STEP}")
print(f"Seeds: {SEEDS}")
print(f"Horyzonty: {HORIZONS}")
print("="*60)

# ============================================
# AUXILIARY FUNCTIONS (hybrid parameters)
# ============================================

def get_hybrid_params(horizon: int, model_type: str = 'lgb', seed: int = 42) -> dict:
    """Hybrid Ollama + Tomek params"""
    
    HORIZON_PARAMS = {
        1:  {'num_leaves': 240, 'lr': 0.028, 'min_child': 77,  'l2': 3.89, 'depth': 10},
        3:  {'num_leaves': 233, 'lr': 0.027, 'min_child': 62,  'l2': 6.22, 'depth': 12},
        10: {'num_leaves': 200, 'lr': 0.025, 'min_child': 100, 'l2': 10.0, 'depth': 14},
        25: {'num_leaves': 350, 'lr': 0.021, 'min_child': 250, 'l2': 20.0, 'depth': 16}
    }
    
    h_params = HORIZON_PARAMS[horizon]
    
    if model_type == 'lgb':
        return {
            'objective': 'regression', 'metric': 'rmse',
            'num_leaves': h_params['num_leaves'],
            'learning_rate': h_params['lr'],
            'n_estimators': 1500,
            'max_depth': h_params['depth'],
            'min_child_samples': h_params['min_child'],
            'subsample': 0.85,
            'colsample_bytree': 0.75,
            'reg_lambda': h_params['l2'],
            'feature_fraction': 0.75,
            'bagging_fraction': 0.85,
            'bagging_freq': 3,
            'random_state': seed,
            'verbose': -1
        }
    
    elif model_type == 'xgb':
        return {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'n_estimators': 1500,
            'max_depth': h_params['depth'],
            'learning_rate': h_params['lr'],
            'min_child_weight': h_params['min_child'],
            'subsample': 0.85,
            'colsample_bytree': 0.75,
            'reg_lambda': h_params['l2'],
            'random_state': seed,
            'verbosity': 0
        }
    
    elif model_type == 'cat':
        return {
            'iterations': 1500,
            'depth': h_params['depth'],
            'learning_rate': h_params['lr'],
            'l2_leaf_reg': h_params['l2'],
            'min_data_in_leaf': h_params['min_child'],
            'subsample': 0.85,
            'random_seed': seed,
            'verbose': False
        }

# ============================================
# SECTION 2: DATA LOAD + IMPUTATION (causal + 0)
# ============================================
print("\n" + "="*60)
print("DATA LOAD + IMPUTATION")
print("="*60)

KAGGLE_ENV = os.path.exists('/kaggle/input')

if KAGGLE_ENV:
    train_path = '/kaggle/input/competitions/ts-forecasting/train.parquet'
    test_path = '/kaggle/input/competitions/ts-forecasting/test.parquet'
else:
    base_dir = Path("..")
    train_path = base_dir / "data" / "train.parquet"
    test_path = base_dir / "data" / "test.parquet"

train_full = pl.read_parquet(train_path)
test_full = pl.read_parquet(test_path)

print(f"✅ Train: {train_full.shape}")
print(f"✅ Test: {test_full.shape}")

# ============================================
# TYPE CONVERSION
# ============================================
print("\n🔄 Type conversion...")

categorical_cols = ['code', 'sub_code', 'sub_category']
for col in categorical_cols:
    if col in train_full.columns:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Categorical))
    if col in test_full.columns:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Categorical))

int_cols = ['horizon', 'ts_index']
for col in int_cols:
    if col in train_full.columns:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Int16))
    if col in test_full.columns:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Int16))

feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
for col in feature_cols:
    if col in train_full.columns and train_full[col].dtype == pl.Float64:
        train_full = train_full.with_columns(pl.col(col).cast(pl.Float32))
    if col in test_full.columns and test_full[col].dtype == pl.Float64:
        test_full = test_full.with_columns(pl.col(col).cast(pl.Float32))

print(f"✅ Type converted: {len(feature_cols)}")

# ============================================
# CAUSAL IMPUTATION (EWMA + forward + 0)
# ============================================
print("\n📌 CAUSAL IMPUTATION (without cross-horizon)")

group_keys = ['code', 'sub_code', 'sub_category', 'horizon']

# For all features
causal_cols = feature_cols.copy()

print(f"Processing {len(causal_cols)} features...")

for c in causal_cols:
    if train_full[c].null_count() > 0 or (c in test_full.columns and test_full[c].null_count() > 0):
        train_full = train_full.with_columns(
            pl.when(pl.col(c).is_null())
            .then(
                pl.coalesce([
                    pl.col(c).ewm_mean(span=20, adjust=False).over(group_keys),
                    pl.col(c).forward_fill().over(group_keys),
                    pl.col(c).ewm_mean(span=10, adjust=False).over(group_keys),
                    pl.lit(0.0)
                ])
            )
            .otherwise(pl.col(c))
            .alias(c)
        )
        
        if c in test_full.columns:
            test_full = test_full.with_columns(
                pl.when(pl.col(c).is_null())
                .then(
                    pl.coalesce([
                        pl.col(c).ewm_mean(span=20, adjust=False).over(group_keys),
                        pl.col(c).forward_fill().over(group_keys),
                        pl.col(c).ewm_mean(span=10, adjust=False).over(group_keys),
                        pl.lit(0.0)
                    ])
                )
                .otherwise(pl.col(c))
                .alias(c)
            )

# Final fallback to 0
for c in feature_cols:
    if train_full[c].null_count() > 0:
        train_full = train_full.with_columns(pl.col(c).fill_null(0).alias(c))
    if c in test_full.columns and test_full[c].null_count() > 0:
        test_full = test_full.with_columns(pl.col(c).fill_null(0).alias(c))

print("✅ Imputation done. NaNs: 0")

# Creating a clean copy (for XGBoost and CatBoost) – no NaNs after imputation
train_czysty = train_full.clone()
test_czysty = test_full.clone()

print("✅ Dirty data (with NaN) for LightGBM")
print("✅ Clean data for XGBoost and CatBoost")

# ============================================
# SEKCJA 3: TIME FEATURES (for all data)
# ============================================
print("\n📌 ADDING TIME FEATURES")

def add_time_features(df: pl.DataFrame, horizon: int = None) -> pl.DataFrame:
    """Adding time features - safe: only ts_index based"""
    df = df.with_columns([
        (pl.col('ts_index') % 200).alias('time_mod_200'),
        (pl.col('ts_index') % 50).alias('time_mod_50'),
        (pl.col('ts_index') % 7).alias('time_mod_7')
    ])
    
    # Horizon-specific EMA (from ts_index, NOT from y_target)
    if horizon is not None:
        if horizon == 1:
            window = 3
        elif horizon == 3:
            window = 5
        elif horizon == 10:
            window = 10
        else:  # 25
            window = 25
        
        df = df.with_columns(
            pl.col('ts_index').rolling_mean(window, min_periods=1).over('code').alias('ts_ema_h')
        ).fill_null(0)
    
    # Cyclical encoding (from ts_index)
    df = df.with_columns([
        ((pl.col('ts_index') / 365.0 * np.pi * 2).sin()).alias('sin_year'),
        ((pl.col('ts_index') / 30.0 * np.pi * 2).sin()).alias('sin_month')
    ]).fill_null(0)
    
    return df

# We add time features to train and test (without horizon-specific EMA – it will be added in a loop)
train_full = add_time_features(train_full, horizon=None)
test_full = add_time_features(test_full, horizon=None)

# the same for clean copies
train_czysty = add_time_features(train_czysty, horizon=None)
test_czysty = add_time_features(test_czysty, horizon=None)

print("✅ Time features added: time_mod_200, time_mod_50, time_mod_7, sin_year, sin_month")

# Add the missing ts_ema_h column to the test data (will be filled with proper values in loop)
test_full = test_full.with_columns(pl.lit(0.0).alias('ts_ema_h'))
test_czysty = test_czysty.with_columns(pl.lit(0.0).alias('ts_ema_h'))
print("✅ ts_ema_h column initialized (will be filled from ts_index in loop)")

# ============================================
# SECTION 4: METRICS
# ============================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom == 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

print("✅ Metrics defined")

# ============================================
# SECTION 5: GENERATING WINDOWS
# ============================================
windows = []
start = MIN_TS
while start + WINDOW_SIZE <= MAX_TS_TRAIN:
    train_end = start + WINDOW_SIZE
    windows.append({
        'train_start': start,
        'train_end': train_end,
    })
    start += STEP

print(f"\n📊 Windows count: {len(windows)}")

# ============================================
# SECTION 6: FEATURES LIST (feature_* + time features)
# ============================================
time_feature_names = ['time_mod_200', 'time_mod_50', 'time_mod_7', 'sin_year', 'sin_month']
all_features = feature_cols + time_feature_names

print(f"✅ Features for models: {len(all_features)} (86 feature_* + 5 time features)")

# ============================================
# SECTION 7: WALK-FORWARD TRENING (1 seed, 3 models)
# ============================================
print("\n" + "="*60)
print("WALK-FORWARD TRENING (1 SEED, 3 MODELE)")
print("="*60)

# Storage for prediction (with window weights)
lgb_predictions = []   # (horizon, seed, w_idx, pred, weight)
xgb_predictions = []
cat_predictions = []

total_combinations = len(HORIZONS) * len(SEEDS) * len(windows)
counter = 0

for w_idx, window in enumerate(windows):
    # Window weight (newer windows = higher weight)
    weight = np.exp(-(len(windows) - 1 - w_idx) * 0.1)
    
    print(f"\n{'='*50}")
    print(f"Okno {w_idx+1}/{len(windows)}: train {window['train_start']}-{window['train_end']}, weight={weight:.4f}")
    print(f"{'='*50}")
    
    for horizon in HORIZONS:
        # Adding horizon-specific EMA to data
        train_h = train_full.filter(
            (pl.col('horizon') == horizon) &
            (pl.col('ts_index') >= window['train_start']) &
            (pl.col('ts_index') <= window['train_end'])
        ).sort('ts_index')
        
        # Adding EMA for the horizon
        if horizon == 1:
            ema_window = 3
        elif horizon == 3:
            ema_window = 5
        elif horizon == 10:
            ema_window = 10
        else:
            ema_window = 25
        
        train_h = train_h.with_columns(
            pl.col('y_target').rolling_mean(ema_window, min_periods=1).over('code').alias('ts_ema_h')
        ).fill_null(0
                   )
        # Same for test (but test doesn't have y_target, so we use 0)
        test_h = test_full.filter(pl.col('horizon') == horizon)
        test_h = test_h.with_columns(pl.lit(0).alias('ts_ema_h'))
        
        for seed in SEEDS:
            counter += 1
            print(f"  [{counter}/{total_combinations}] H={horizon}, seed={seed}...", end=" ")
            
            # ========== LIGHTGBM ==========
            if len(train_h) >= 100:
                params = get_hybrid_params(horizon, 'lgb', seed)
                model = lgb.LGBMRegressor(**params)
                
                X = train_h.select(all_features + ['ts_ema_h']).to_numpy()
                y = train_h.select('y_target').to_numpy().ravel()
                w = train_h.select('weight').to_numpy().ravel()
                
                model.fit(X, y, sample_weight=w)
                
                X_test = test_h.select(all_features + ['ts_ema_h']).to_numpy()
                pred = model.predict(X_test)
                lgb_predictions.append((horizon, seed, w_idx, pred, weight))
                print("LGB ✓", end=" ")
            
            # ========== XGBOOST ==========
            train_h_clean = train_czysty.filter(
                (pl.col('horizon') == horizon) &
                (pl.col('ts_index') >= window['train_start']) &
                (pl.col('ts_index') <= window['train_end'])
            ).sort('ts_index')

            # Add EMA for clean data
            train_h_clean = train_h_clean.with_columns(
                pl.col('y_target').rolling_mean(ema_window, min_periods=1).over('code').alias('ts_ema_h')
            ).fill_null(0)
            
            if len(train_h_clean) >= 100:
                params = get_hybrid_params(horizon, 'xgb', seed)
                model = xgb.XGBRegressor(**params)
                
                X = train_h_clean.select(all_features + ['ts_ema_h']).to_numpy()
                y = train_h_clean.select('y_target').to_numpy().ravel()
                w = train_h_clean.select('weight').to_numpy().ravel()
                
                model.fit(X, y, sample_weight=w, verbose=False)
                
                X_test = test_czysty.select(all_features + ['ts_ema_h']).to_numpy()
                pred = model.predict(X_test)
                xgb_predictions.append((horizon, seed, w_idx, pred, weight))
                print("XGB ✓", end=" ")
            
            # ========== CATBOOST ==========
            if len(train_h_clean) >= 100:
                params = get_hybrid_params(horizon, 'cat', seed)
                model = CatBoostRegressor(**params)
                
                X = train_h_clean.select(all_features + ['ts_ema_h'] + categorical_cols).to_pandas()
                y = train_h_clean.select('y_target').to_numpy().ravel()
                w = train_h_clean.select('weight').to_numpy().ravel()
                
                model.fit(X, y, sample_weight=w, cat_features=categorical_cols, verbose=False)
                
                X_test = test_czysty.select(all_features + ['ts_ema_h'] + categorical_cols).to_pandas()
                pred = model.predict(X_test)
                cat_predictions.append((horizon, seed, w_idx, pred, weight))
                print("CAT ✓", end=" ")
            
            print()

print(f"\n✅ Trained:")
print(f"   LightGBM: {len(lgb_predictions)} models")
print(f"   XGBoost: {len(xgb_predictions)} models")
print(f"   CatBoost: {len(cat_predictions)} models")

# ============================================
# SEKCJA 8: ENSEMBLE (weigthed average)
# ============================================
print("\n" + "="*60)
print("ENSEMBLE PREDICTIONS (weighted by window)")
print("="*60)

final_preds = {}

for horizon in HORIZONS:
    all_preds = []
    all_weights = []
    
    for (h, seed, w_idx, pred, weight) in lgb_predictions:
        if h == horizon:
            all_preds.append(pred)
            all_weights.append(weight)
    
    for (h, seed, w_idx, pred, weight) in xgb_predictions:
        if h == horizon:
            all_preds.append(pred)
            all_weights.append(weight)
    
    for (h, seed, w_idx, pred, weight) in cat_predictions:
        if h == horizon:
            all_preds.append(pred)
            all_weights.append(weight)
    
    if all_preds:
        # Weighted average
        all_preds = np.array(all_preds)
        all_weights = np.array(all_weights)
        final_pred = np.average(all_preds, axis=0, weights=all_weights)
        final_preds[horizon] = final_pred
        print(f"Horizon {horizon}: {len(all_preds)} models → weighted ensemble")
# ============================================
# SEKCJA 8.5: DIAGNOSTICS & METRICS
# ============================================
print("\n" + "="*60)
print("DIAGNOSTICS & METRICS (per horizon)")
print("="*60)

for horizon in HORIZONS:
    print(f"\n{'='*60}")
    print(f"HORIZON: {horizon}")
    print(f"{'='*60}")
    
    # Get training data for this horizon (from last window - most representative)
    train_h = train_full.filter(
        (pl.col('horizon') == horizon) &
        (pl.col('ts_index') >= windows[-1]['train_start']) &
        (pl.col('ts_index') <= windows[-1]['train_end'])
    ).sort('ts_index')
    
    if len(train_h) == 0:
        print(f"  No training data for H={horizon}")
        continue
    
    # Train a final model for diagnostics (using best params)
    params = get_hybrid_params(horizon, 'lgb', SEEDS[0])
    model_diag = lgb.LGBMRegressor(**params)
    
    X_train = train_h.select(all_features + ['ts_ema_h']).to_numpy()
    y_train = train_h['y_target'].to_numpy().ravel()
    w_train = train_h['weight'].to_numpy().ravel()
    
    model_diag.fit(X_train, y_train, sample_weight=w_train)
    
    # Training metrics
    y_pred_train = model_diag.predict(X_train)
    train_score = weighted_rmse_score(y_train, y_pred_train, w_train)
    train_pearson = np.corrcoef(y_train, y_pred_train)[0, 1]
    train_mae = np.mean(np.abs(y_train - y_pred_train))
    train_r2 = 1 - np.sum((y_train - y_pred_train)**2) / np.sum((y_train - np.mean(y_train))**2)
    
    print(f"\n  📊 TRAINING METRICS (last window):")
    print(f"    Weighted RMSE: {train_score:.6f}")
    print(f"    Pearson: {train_pearson:.6f}")
    print(f"    MAE: {train_mae:.6f}")
    print(f"    R²: {train_r2:.6f}")
    print(f"    y_pred std: {y_pred_train.std():.4f} (target std: {y_train.std():.4f})")
    
    # Test predictions (from ensemble)
    if horizon in final_preds:
        test_preds = final_preds[horizon]
        test_h = test_full.filter(pl.col('horizon') == horizon)
        
        print(f"\n  📊 TEST PREDICTIONS (ensemble):")
        print(f"    Samples: {len(test_preds):,}")
        print(f"    Mean: {np.mean(test_preds):.6f}")
        print(f"    Std: {np.std(test_preds):.6f}")
        print(f"    Range: [{np.min(test_preds):.4f}, {np.max(test_preds):.4f}]")

# ============================================
# TIMER - END
# ============================================
notebook_end_time = time.time()
total_time_seconds = notebook_end_time - notebook_start_time
total_time_minutes = total_time_seconds / 60
total_time_hours = total_time_minutes / 60

print("\n" + "="*80)
print("TIMING SUMMARY")
print("="*80)
print(f"⏱️ Started:  {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⏱️ Total execution time: {total_time_seconds:.2f} seconds ({total_time_minutes:.2f} minutes / {total_time_hours:.2f} hours)")
print("="*80)
# ============================================
# SECTION 9: SUBMISSION
# ============================================
print("\n" + "="*60)
print("SUBMISSION")
print("="*60)

all_ids = []
all_preds = []

for horizon in HORIZONS:
    test_h = test_full.filter(pl.col('horizon') == horizon)
    ids = test_h.select('id').to_numpy().ravel()
    
    if horizon in final_preds:
        preds = final_preds[horizon]
        all_ids.extend(ids)
        all_preds.extend(preds)
    else:
        print(f"⚠️ No prediction for horizon{horizon}")

submission_df = pl.DataFrame({
    'id': all_ids,
    'prediction': all_preds
})

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = f"/kaggle/working/walkforward_hybrid_{timestamp}.csv"
submission_df.write_csv(submission_path)

print(f"✅ Submission saved: {submission_path}")
print(f"📊 Shape: {submission_df.shape}")
print(f"📊 Prediction range: [{np.min(all_preds):.4f}, {np.max(all_preds):.4f}]")
print(submission_df.head())

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*80)
print("WALK-FORWARD HYBRID - PODSUMOWANIE")
print("="*80)
print(f"✅ Windows: {len(windows)}")
print(f"✅ Seeds: {SEEDS}")
print(f"✅ LightGBM: {len(lgb_predictions)} models")
print(f"✅ XGBoost: {len(xgb_predictions)} models")
print(f"✅ CatBoost: {len(cat_predictions)} models")
print(f"✅ Ensemble: weighted by window (decay=0.1)")
print(f"✅ Time features: {time_feature_names} + ts_ema_h")
print(f"✅ Imputation: causal (EWMA + forward + 0)")
print(f"✅ Submission: {submission_path}")
print("="*80)

⏱️ Notebook started at: 2026-04-09 19:59:36
ENVIRONMENT VERSIONS
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
LightGBM version: 4.6.0
Polars version: 1.35.2
NumPy version: 2.0.2
CONFIGURATION OF WALK-FORWARD (1 SEED + HYBRID PARAMS)
Window size: 500, Step: 100
Seeds: [42]
Horyzonty: [1, 3, 10, 25]

DATA LOAD + IMPUTATION
✅ Train: (5337414, 94)
✅ Test: (1447107, 92)

🔄 Type conversion...
✅ Type converted: 86

📌 CAUSAL IMPUTATION (without cross-horizon)
Processing 86 features...
✅ Imputation done. NaNs: 0
✅ Dirty data (with NaN) for LightGBM
✅ Clean data for XGBoost and CatBoost

📌 ADDING TIME FEATURES
✅ Time features added: time_mod_200, time_mod_50, time_mod_7, sin_year, sin_month
✅ ts_ema_h column initialized (will be filled from ts_index in loop)
✅ Metrics defined

📊 Windows count: 32
✅ Features for models: 91 (86 feature_* + 5 time features)

WALK-FORWARD TRENING (1 SEED, 3 MODELE)

Okno 1/32: train 1-501, weight=0.0450
  [1/128] H=1, seed=42... LGB ✓ XGB ✓ CAT